# Stage 1 — Evaluate the pre-trained base model

Evaluates `./vi-smollm-135m-pretrain/` (checkpoints or final model).

**Prerequisites:** tokenizer + Stage 1 pretrain (`finetuning/train.py --mode pretrain`).

**What this measures** (a 135M base model cannot answer questions — judge it on language modelling, not facts):

1. Config sanity
2. Perplexity on the plan's eval set `VTSNLP/vietnamese_curated_dataset` (docs/stage1_pretrain.md), plus held-out CulturaX and curated sentences
3. Fact knowledge as multiple choice (log-likelihood ranking)
4. Qualitative continuations (greedy and sampled)
5. Repetition rate of generations
6. Comparison across checkpoints (set `CHECKPOINTS`)

In [1]:
import os
import sys

def _find_repo_root():
    d = os.getcwd()
    while not os.path.isdir(os.path.join(d, "finetuning")):
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Run this notebook from inside the ViRL-SmolLM2-135M repo")
        d = parent
    return d

REPO_ROOT = _find_repo_root()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

In [2]:
import glob
import re
import torch
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

from utils.eval_utils import compute_perplexity

PRETRAIN_DIR = "./vi-smollm-135m-pretrain"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

def list_checkpoints():
    cks = glob.glob(f"{PRETRAIN_DIR}/checkpoint-*")
    return sorted(cks, key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))

available = list_checkpoints()
print("Available checkpoints:", available)

# Final model if saved (has model.safetensors at the top level), else the latest checkpoint.
if os.path.exists(f"{PRETRAIN_DIR}/model.safetensors"):
    MODEL_DIR = PRETRAIN_DIR
else:
    assert available, f"No checkpoint in {PRETRAIN_DIR} — run Stage 1 first"
    MODEL_DIR = available[-1]

# Set to a list of paths to compare checkpoints in section 6; defaults to all of them.
CHECKPOINTS = available

def load(path):
    return AutoModelForCausalLM.from_pretrained(path, dtype=DTYPE).to(DEVICE).eval()

tokenizer = PreTrainedTokenizerFast.from_pretrained("./vi_smollm_tokenizer")
model = load(MODEL_DIR)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Loaded {MODEL_DIR} ({n_params:.1f}M params) on {DEVICE}")

/home/thuandn/Repository/ViRL-SmolLM2-135M/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available checkpoints: ['./vi-smollm-135m-pretrain/checkpoint-41667', './vi-smollm-135m-pretrain/checkpoint-80000']


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2999.55it/s]


✅ Loaded ./vi-smollm-135m-pretrain (124.6M params) on cuda


## 1. Config

In [3]:
assert model.config.vocab_size == 32000, f"vocab_size={model.config.vocab_size}, expected 32000"
assert model.config.max_position_embeddings >= 2048
assert len(tokenizer) == model.config.vocab_size, "tokenizer / model vocab mismatch"
print(f"✅ Config OK — vocab_size={model.config.vocab_size}, max_pos={model.config.max_position_embeddings}")

✅ Config OK — vocab_size=32000, max_pos=2048


## 2. Perplexity

- **`VTSNLP/vietnamese_curated_dataset`** — the Stage 1 eval set defined in `docs/stage1_pretrain.md` (general Vietnamese text, never used in training). **This is the headline number**, reported overall and per domain. Needs network access.
- **Held-out CulturaX** — shard `vi_part_00004` (training uses `vi_part_0000[0-3]`): in-distribution reference.
- **Curated sentences** — tiny and noisy, rough signal only.

In [4]:
eval_texts = [
    "Việt Nam là một quốc gia ở Đông Nam Á.",
    "Thủ đô của Việt Nam là Hà Nội.",
    "Phở bò là một món ăn truyền thống của người Việt Nam.",
    "Ngôn ngữ lập trình Python rất phổ biến trong lĩnh vực trí tuệ nhân tạo.",
    "Đồng bằng sông Cửu Long là vùng sản xuất lúa lớn nhất cả nước.",
]
for t in eval_texts:
    print(f"{compute_perplexity(model, tokenizer, [t]):9.2f}  {t}")
print(f"\nCurated PPL (token-weighted): {compute_perplexity(model, tokenizer, eval_texts):.2f}")

     4.07  Việt Nam là một quốc gia ở Đông Nam Á.
    16.97  Thủ đô của Việt Nam là Hà Nội.
     4.78  Phở bò là một món ăn truyền thống của người Việt Nam.
     8.32  Ngôn ngữ lập trình Python rất phổ biến trong lĩnh vực trí tuệ nhân tạo.
     7.23  Đồng bằng sông Cửu Long là vùng sản xuất lúa lớn nhất cả nước.

Curated PPL (token-weighted): 7.03


In [5]:
EVAL_DOCS = 300
EVAL_MAX_LEN = 512

def load_vtsnlp(n=EVAL_DOCS):
    """Plan eval set. Stream the first n docs; keep domain for a per-domain breakdown."""
    from datasets import load_dataset
    ds = load_dataset("VTSNLP/vietnamese_curated_dataset", split="train", streaming=True)
    rows = [x for _, x in zip(range(n), ds)]
    return [r["text"] for r in rows], [r["domain"] for r in rows]

def load_culturax_heldout(n=EVAL_DOCS):
    from datasets import load_dataset
    ds = load_dataset("uonlp/CulturaX", "vi", data_files="vi/vi_part_00004.parquet",
                      split="train", streaming=True)
    return [x["text"] for _, x in zip(range(n), ds)]

try:
    vtsnlp_texts, vtsnlp_domains = load_vtsnlp()
    print(f"Loaded {len(vtsnlp_texts)} VTSNLP docs")
except Exception as e:
    vtsnlp_texts = vtsnlp_domains = None
    print(f"⚠️  VTSNLP unavailable ({type(e).__name__}: {e}) — skipping")

try:
    heldout_texts = load_culturax_heldout()
    print(f"Loaded {len(heldout_texts)} held-out CulturaX docs")
except Exception as e:
    heldout_texts = None
    print(f"⚠️  CulturaX held-out unavailable ({type(e).__name__}: {e}) — skipping")

def ppl(m, texts):
    return compute_perplexity(m, tokenizer, texts, max_length=EVAL_MAX_LEN)

if vtsnlp_texts:
    print(f"\n🎯 VTSNLP PPL ({EVAL_MAX_LEN} tokens/doc): {ppl(model, vtsnlp_texts):.2f}   (doc target < 15)")
    from collections import defaultdict
    by = defaultdict(list)
    for t, d in zip(vtsnlp_texts, vtsnlp_domains):
        by[d].append(t)
    print("Per domain:")
    for d, ts in sorted(by.items(), key=lambda kv: -len(kv[1])):
        print(f"  {d:<28} n={len(ts):<4} PPL={ppl(model, ts):.2f}")
if heldout_texts:
    print(f"\nCulturaX held-out PPL: {ppl(model, heldout_texts):.2f}")

Loaded 300 VTSNLP docs
Loaded 300 held-out CulturaX docs

🎯 VTSNLP PPL (512 tokens/doc): 54.81   (doc target < 15)
Per domain:
  Jobs_and_Education           n=75   PPL=70.38
  People_and_Society           n=54   PPL=43.60
  Pets_and_Animals             n=46   PPL=88.15
  Science                      n=29   PPL=63.80
  Travel_and_Transportation    n=23   PPL=51.55
  News                         n=14   PPL=51.45
  Arts_and_Entertainment       n=9    PPL=34.85
  Sensitive_Subjects           n=8    PPL=42.75
  Books_and_Literature         n=7    PPL=55.63
  Business_and_Industrial      n=5    PPL=59.94
  Internet_and_Telecom         n=4    PPL=60.35
  Food_and_Drink               n=4    PPL=52.34
  Sports                       n=4    PPL=48.54
  Law_and_Government           n=4    PPL=40.72
  Computers_and_Electronics    n=3    PPL=52.25
  Beauty_and_Fitness           n=3    PPL=31.95
  Autos_and_Vehicles           n=3    PPL=39.05
  Health                       n=3    PPL=35.67
  Finance

## 3. Fact knowledge (multiple choice)

Scores `log P(answer | prompt)` for each candidate and checks whether the correct one ranks first. Chance level is 1/len(choices). A 135M base model is expected to be near chance on many items.

In [6]:
FACTS = [
    ("Thủ đô của Việt Nam là", "Hà Nội", ["Thành phố Hồ Chí Minh", "Đà Nẵng", "Huế"]),
    ("Thành phố lớn nhất Việt Nam là", "Thành phố Hồ Chí Minh", ["Hà Nội", "Cần Thơ", "Hải Phòng"]),
    ("Đồng bằng sông Cửu Long nằm ở miền", "Nam", ["Bắc", "Trung", "Tây"]),
    ("Tết Nguyên Đán là dịp lễ truyền thống của người", "Việt", ["Mỹ", "Nga", "Pháp"]),
    ("Món ăn nổi tiếng của Hà Nội là", "phở", ["sushi", "pizza", "kimchi"]),
    ("Quốc ca của Việt Nam là bài", "Tiến quân ca", ["Quốc tế ca", "Marseillaise", "Kimigayo"]),
]

@torch.no_grad()
def continuation_logprob(m, prompt, answer):
    p_ids = tokenizer(prompt)["input_ids"]
    full = tokenizer(prompt + " " + answer)["input_ids"]
    ids = torch.tensor([full], device=DEVICE)
    logp = torch.log_softmax(m(ids).logits[0, :-1].float(), dim=-1)
    tgt = ids[0, 1:]
    per_tok = logp.gather(1, tgt[:, None])[:, 0]
    return per_tok[len(p_ids) - 1:].sum().item()

def eval_facts(m, verbose=True):
    correct = 0
    for prompt, gold, wrong in FACTS:
        scores = {c: continuation_logprob(m, prompt, c) for c in [gold] + wrong}
        ranked = sorted(scores, key=scores.get, reverse=True)
        ok = ranked[0] == gold
        correct += ok
        if verbose:
            print(f"{'✅' if ok else '❌'} {prompt} → top: {ranked[0]!r} (gold {gold!r}, rank {ranked.index(gold)+1}/{len(ranked)})")
    return correct / len(FACTS)

acc = eval_facts(model)
print(f"\nFact accuracy: {acc:.0%}  (chance ≈ 25%)")

✅ Thủ đô của Việt Nam là → top: 'Hà Nội' (gold 'Hà Nội', rank 1/4)
❌ Thành phố lớn nhất Việt Nam là → top: 'Hà Nội' (gold 'Thành phố Hồ Chí Minh', rank 2/4)
❌ Đồng bằng sông Cửu Long nằm ở miền → top: 'Bắc' (gold 'Nam', rank 3/4)
✅ Tết Nguyên Đán là dịp lễ truyền thống của người → top: 'Việt' (gold 'Việt', rank 1/4)
✅ Món ăn nổi tiếng của Hà Nội là → top: 'phở' (gold 'phở', rank 1/4)
✅ Quốc ca của Việt Nam là bài → top: 'Tiến quân ca' (gold 'Tiến quân ca', rank 1/4)

Fact accuracy: 67%  (chance ≈ 25%)


## 4. Qualitative continuations

Base model: use completion-style prompts, not questions.

In [7]:
PROMPTS = [
    "Thủ đô của Việt Nam là",
    "Hôm nay trời đẹp, tôi quyết định",
    "Phở là một món ăn",
    "Theo báo cáo của Bộ Giáo dục và Đào tạo,",
]

@torch.no_grad()
def complete(m, prompt, n_new=60, **gen):
    x = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    x.pop("token_type_ids", None)
    out = m.generate(**x, max_new_tokens=n_new, pad_token_id=tokenizer.pad_token_id, **gen)
    return tokenizer.decode(out[0], skip_special_tokens=True)

torch.manual_seed(0)
for p in PROMPTS:
    print("PROMPT:", p)
    print("  greedy :", complete(model, p, do_sample=False, repetition_penalty=1.2).replace("\n", " ⏎ "))
    for i in range(2):
        print(f"  sample{i}:", complete(model, p, do_sample=True, temperature=0.7, top_p=0.9).replace("\n", " ⏎ "))
    print()

PROMPT: Thủ đô của Việt Nam là
  greedy : Thủ đô của Việt Nam là một trong những điểm đến hấp dẫn nhất thế giới. ⏎ Chủ đề: Thủ đô của Việt Nam là một trong những điểm đến hấp dẫn nhất thế giới ⏎ 12-01-2019, 10:00 AM #1 ⏎ Hà Nội có nhiều địa danh nổi tiếng như Chùa Cầu, chùa Linh Ứng
  sample0: Thủ đô của Việt Nam là một trong những quốc gia có nền kinh tế phát triển nhất thế giới. Tuy nhiên, Việt Nam vẫn là một trong những quốc gia có nền kinh tế phát triển nhất thế giới. ⏎ Cùng với sự phát triển của kinh tế, xã hội, nhu cầu sử dụng hàng hóa của người dân tăng
  sample1: Thủ đô của Việt Nam là một thành phố hiện đại với nhiều tòa nhà cao tầng, trung tâm thương mại, các trung tâm thương mại, khu vui chơi giải trí, trung tâm mua sắm, trung tâm mua sắm… ⏎ Trong khi đó, những thành phố lớn như Hà Nội, Đà Nẵng, Đà Nẵng… vẫn có

PROMPT: Hôm nay trời đẹp, tôi quyết định
  greedy : Hôm nay trời đẹp, tôi quyết định đi du lịch ở đâu? - Du Lịch Việt ⏎ Trang chủ » Tin tức » Ngày mai mưa gió – Mưa 

## 5. Repetition rate

Fraction of repeated word 3-grams in samples (lower is better). Greedy without a repetition penalty exposes degeneration.

In [8]:
def rep_ngram_rate(text, n=3):
    w = text.split()
    grams = [tuple(w[i:i+n]) for i in range(len(w) - n + 1)]
    return 0.0 if not grams else 1 - len(set(grams)) / len(grams)

def eval_repetition(m, prompts=PROMPTS, n_new=100):
    torch.manual_seed(0)
    g = [rep_ngram_rate(complete(m, p, n_new, do_sample=False)) for p in prompts]
    s = [rep_ngram_rate(complete(m, p, n_new, do_sample=True, temperature=0.7, top_p=0.9)) for p in prompts]
    return sum(g) / len(g), sum(s) / len(s)

rg, rs = eval_repetition(model)
print(f"3-gram repetition — greedy: {rg:.1%}, sampled: {rs:.1%}")

3-gram repetition — greedy: 53.9%, sampled: 25.4%


## 6. Compare checkpoints

Same metrics across `CHECKPOINTS` (edit the list above to restrict). Loss should fall and held-out PPL should drop as steps increase.

In [9]:
import pandas as pd

rows = []
for path in CHECKPOINTS:
    m = load(path)
    row = {"checkpoint": os.path.basename(path),
           "curated_ppl": compute_perplexity(m, tokenizer, eval_texts),
           "fact_acc": eval_facts(m, verbose=False)}
    if vtsnlp_texts:
        row["vtsnlp_ppl"] = ppl(m, vtsnlp_texts)
    if heldout_texts:
        row["culturax_ppl"] = ppl(m, heldout_texts)
    row["rep3_greedy"], row["rep3_sampled"] = eval_repetition(m)
    rows.append(row)
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

pd.DataFrame(rows).round(3)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2925.06it/s]


,checkpoint,curated_ppl,fact_acc,vtsnlp_ppl,culturax_ppl,rep3_greedy,rep3_sampled
0,checkpoint-41667,7.533,0.667,56.229,20.873,0.470,0.190
1,checkpoint-80000,7.031,0.667,54.808,20.414,0.539,0.254


## Reading the results

- **VTSNLP PPL** is the main signal (the plan's eval set); the doc target of `< 15` is indicative only and depends on tokens seen. A 135M model on ~half a run is typically well above that.
- **Fact accuracy** near chance and **repetition** under greedy decoding are normal for a base model; both improve after SFT (Stage 2).
- Training loss curve: `vi-smollm-135m-pretrain/checkpoint-*/trainer_state.json` → `log_history`.